<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Fundamentals of Generative AI</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 2b: Amazon Bedrock with LangChain</h2>
</div>

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents -->
<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2874A6; border-bottom: 1px solid #2874A6; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2E86C1; font-weight: bold; text-decoration: none;">1. Installation and API calls</a></p>
    <p><a href="#section2" style="color: #2E86C1; font-weight: bold; text-decoration: none;">2. Starting the conversation</a></p>
    <ul style="margin-top: 0; padding-left: 30px;">
        <li><a href="#section2-1" style="color: #3498DB; text-decoration: none;">2.1 First message</a></li>
        <li><a href="#section2-2" style="color: #3498DB; text-decoration: none;">2.2 Second message</a></li>
        <li><a href="#section2-3" style="color: #3498DB; text-decoration: none;">2.3 Third message</a></li>
        <li><a href="#section2-4" style="color: #3498DB; text-decoration: none;">2.4 The last message</a></li>
    </ul>
    <p><a href="#section3" style="color: #2E86C1; font-weight: bold; text-decoration: none;">3. Quiz Questions</a></p>
</div>

[__LangChain__](https://python.langchain.com/docs/introduction/) is a popular open source framework to develop applications with Large Language Models. Recent versions of LangChain started to support Amazon Bedrock models.

In this notebook, we will build __a simple chatbot__ using Amazon's __Amazon Nova Micro__ with Amazon Bedrock.

### In this lab, we will cover topics such as:
- Setting up and accessing the Bedrock service using boto3
- Exploring available Large Language Models (LLMs) in Bedrock
- Performing Bedrock API calls with various customization options
- Understanding and manipulating model parameters for text generation

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">1. Installation and API calls</h2>
</div>
In this section, we'll set up the necessary libraries and establish connections to Amazon Bedrock services.

Installing a recent version of LangChain.

In [ ]:
!pip install --no-deps -q -r ../requirements.txt

Code that will suppress deprecation warnings.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

We import the Bedrock module first and use the __Amazon Nova Micro__ from it. We can pass model parameters at this point. For example we set the temperature to zero and maximum number of tokens in the text response to 500.

In [ ]:
import boto3
from langchain_aws import ChatBedrockConverse

session = boto3.session.Session()

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    temperature=0,
    max_tokens=500,
)

We can also set a certain prompt template. Below, we create a slightly different version of the default template.

In [ ]:
from langchain.prompts.prompt import PromptTemplate

template = """The following is a friendly conversation between a human and an AI. \
If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Question: {input}
Answer:"""

prompt_template = PromptTemplate(
    input_variables=["history", "input"], template=template
)

Once we have the LLM and the prompt template, we can start a conversation chain. Inside the function, we provide the LLM and set a few other parameters. 
* __verbose__ prints the conversation history during the chat
* __memory__ is responsible for storing the conversation history
* Inside __ConversationBufferMemory()__, we can also set different names for the AI and user through __ai_prefix__ and __human_prefix__

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

# Store for session histories
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """
    Get or create a chat message history for a given session ID.
    This is the factory function required by RunnableWithMessageHistory.
    """
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Create the prompt template that matches the original ConversationChain behavior
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "The following is a friendly conversation between a human and an AI. "
               "If the AI does not know the answer to a question, it truthfully says it does not know."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Create the basic chain (prompt + llm)
chain = prompt_template | llm

# Wrap the chain with message history using RunnableWithMessageHistory
runnable_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Create a complete wrapper that includes all attributes the notebook expects
class CompleteConversationWrapper:
    """Complete wrapper that maintains all attributes and methods for notebook compatibility"""

    def __init__(self, runnable_with_history, prompt_template, session_id="notebook_session"):
        self.runnable = runnable_with_history
        self.session_id = session_id

        # Create a simple object to hold the template string for compatibility
        class PromptTemplateCompat:
            def __init__(self, template_str):
                self.template = template_str

        # Extract the template string from the ChatPromptTemplate for display
        template_str = """The following is a friendly conversation between a human and an AI. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Question: {input}
Answer:"""

        self.prompt = PromptTemplateCompat(template_str)

    def predict(self, input):
        """Predict method that matches the original ConversationChain.predict() interface"""
        response = self.runnable.invoke(
            {"input": input},
            config={"configurable": {"session_id": self.session_id}}
        )

        # Extract content from the response
        if hasattr(response, 'content'):
            return response.content
        else:
            return str(response)

# Create the conversation object that works with existing notebook code
conversation = CompleteConversationWrapper(runnable_with_history, prompt_template)

print("✓ RunnableWithMessageHistory conversation created successfully!")
print("✓ No deprecation warnings - using official recommended approach")
print("✓ All attributes (including .prompt) available for notebook compatibility")


Let's print out our conversation template. This is a general template for conversation.

In [ ]:
from IPython.display import Markdown, display

default_prompt_template = conversation.prompt.template

Markdown(default_prompt_template)

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">2. Starting the conversation</h2>
</div>
Let's start the conversation!  We send our message by calling the __predict()__ function with our text. As we set the __verbose__ parameter __true__ earlier, history of the conversation will be printed out first. Then, we will see the response of the chatbot.

<!-- Subsection Header -->
<div id="section2-1" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">2.1 First message</h3>
</div>

Let's start with asking the AI if they are familiar with machine learning.

In [ ]:
Markdown(conversation.predict(input="Hello! Are you familiar with Machine Learning?"))

<!-- Subsection Header -->
<div id="section2-2" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">2.2 Second message</h3>
</div>

Next, we are specifically interested in LLMs. Let's ask about them.

In [ ]:
Markdown(
    conversation.predict(
        input="Can you list a few applications of Large Language Models?"
    )
)

<!-- Subsection Header -->
<div id="section2-3" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">2.3 Third message</h3>
</div>

We ask about some recent developments in the field to learn more.

In [ ]:
Markdown(conversation.predict(input="What are some recent developments in LLMs?"))

<!-- Subsection Header -->
<div id="section2-4" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">2.4 The last message</h3>
</div>

At the end, we thank and end the conversation.

In [ ]:
Markdown(conversation.predict(input="Nice. Thanks."))

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">3. Quiz Questions</h2>
</div>

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Knowledge Assessment</h4>
        <p>Well done on completing the lab! Now, it's time for a brief knowledge assessment.</p>
        <p>Answer the following questions to test your understanding of using MLLMs for inference.</p>
    </div>
</div>

In [ ]:
import sys
sys.path.append('..')
from mlu_utils.quiz_questions import lab2b_question1

lab2b_question1.display()

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Conclusion</h3>
    <p>In this lab, you have:</p>
    <ul>
        <li>Set up and configured Amazon Bedrock with LangChain</li>
        <li>Created a custom prompt template for conversations</li>
        <li>Built a conversational AI using Amazon Nova Micro</li>
        <li>Learned how to maintain conversation context with memory components</li>
    </ul>
    <h4 style="color: #2874A6; margin-top: 15px;">Additional Resources</h4>
    <ul>
        <li><a href="https://python.langchain.com/docs/introduction/">LangChain Documentation</a></li>
        <li><a href="https://aws.amazon.com/bedrock/">Amazon Bedrock</a></li>
    </ul>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!